In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('train.csv')
df.shape

(404290, 6)

In [3]:
df.head(3)

,id,qid1,qid2,question1,question2,is_duplicate
0,0,1,2,What is the step by step guide to invest in sh...,What is the step by step guide to invest in sh...,0
1,1,3,4,What is the story of Kohinoor (Koh-i-Noor) Dia...,What would happen if the Indian government sto...,0
2,2,5,6,How can I increase the speed of my internet co...,How can Internet speed be increased by hacking...,0


In [4]:
smalldf = df.sample(50000).reset_index(drop = True)
smalldf.shape

(50000, 6)

In [5]:
smalldf.head(3)

,id,qid1,qid2,question1,question2,is_duplicate
0,212144,206408,54675,How one can define success?,What do you define as success?,1
1,389944,289407,522437,Which ones are considered Gustav Mahler's best...,How wealthy was Gustav Mahler by the end of hi...,0
2,300177,422935,107891,Why is everyone trying to kill me?,"Someone is trying to kill me, what should I do?",0


In [6]:
questions = smalldf[['question1','question2']]
questions.shape

(50000, 2)

In [7]:
questions.head(3)

,question1,question2
0,How one can define success?,What do you define as success?
1,Which ones are considered Gustav Mahler's best...,How wealthy was Gustav Mahler by the end of hi...
2,Why is everyone trying to kill me?,"Someone is trying to kill me, what should I do?"


In [8]:
questions.isnull().sum()

question1    1
question2    1
dtype: int64

In [9]:
questions = questions.dropna()
questions.isnull().sum()

question1    0
question2    0
dtype: int64

In [10]:
questions.shape

(49998, 2)

In [11]:
import re

def clean_text(text):
    text = text.lower()                             # Convert text to lower-case
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)      # Remove punctuations
    text = re.sub(r"\s+", " ", text).strip()        # Removes any extra blank space
    return text

questions['question1'] = questions['question1'].apply(clean_text)
questions['question2'] = questions['question2'].apply(clean_text)

In [12]:
sentences = (questions['question1'].str.split().tolist() +      # Combining all the questions into a list of lists of tokenized questions/sentences
             questions['question2'].str.split().tolist())       # This creates the expected input form for the Word2Vec model training

In [13]:
!pip install gensim

In [14]:
from gensim.models import Word2Vec
# Initializing the Word2Vec model
w2v = Word2Vec(sentences, vector_size = 300, window = 6, min_count = 3, workers = 4, sg = 0, negative = 10, epochs = 30)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [15]:
def sentence_vector(tokens, model, vector_size):     # Defining function to create embeddings given a list of tokens (sentence)
    valid_words = [model.wv[word]for word in tokens if word in model.wv]
    if len(valid_words) ==0:
        return np.zeros(vector_size)
    return np.mean(valid_words, axis = 0)


q1_embeddings = np.array([sentence_vector(q.split(), w2v, 300)     # Creating embeddings for 'question1' column
                          for q in questions['question1']])
q2_embeddings = np.array([sentence_vector(q.split(), w2v, 300)     # Creating embeddings for 'question2' column
                         for q in questions['question2']])

In [16]:
smalldf.head(3)       # "smalldf" contains the labels for the respective questions

,id,qid1,qid2,question1,question2,is_duplicate
0,212144,206408,54675,How one can define success?,What do you define as success?,1
1,389944,289407,522437,Which ones are considered Gustav Mahler's best...,How wealthy was Gustav Mahler by the end of hi...,0
2,300177,422935,107891,Why is everyone trying to kill me?,"Someone is trying to kill me, what should I do?",0


In [17]:
smalldf.shape       # But "smalldf" is of size 50,000 due to the null value not being deleted from the DataFrame

(50000, 6)

In [18]:
smalldf = smalldf.dropna()     # Removing the null row from "smalldf" Dataframe
smalldf.shape                  # to match the number of rows of X (that is equal to the number of rows/entries in the "questions" DataFrame)

(49998, 6)

In [19]:
X = np.hstack([q1_embeddings, q2_embeddings])    # Horizontally concatenates the two matrices along columns. This would work as the datapoints for the training and testing of models
y = smalldf['is_duplicate'].values         # Extracts the target variable (0/1) as a vector.This would work as the label for the training and testing of models

In [20]:
from sklearn.model_selection import train_test_split       # Spliting the questions and its corresponding labels into Training and Testing sets

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [21]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

svm = SVC(kernel = 'rbf', C = 1.0)
svm.fit(X_train, y_train)

SVC()

In [22]:
y_pred = svm.predict(X_test)
print("SVM Model Accuracy: ", accuracy_score(y_test, y_pred))

SVM Model Accuracy:  0.7723


In [23]:
print("\n Classification Report: ")
print(classification_report(y_test, y_pred))


 Classification Report: 
              precision    recall  f1-score   support

           0       0.79      0.88      0.83      6366
           1       0.74      0.58      0.65      3634

    accuracy                           0.77     10000
   macro avg       0.76      0.73      0.74     10000
weighted avg       0.77      0.77      0.77     10000



In [24]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter = 2000)
logreg.fit(X_train, y_train)

LogisticRegression(max_iter=2000)

In [26]:
y_pred_logreg = logreg.predict(X_test)
print("Logistic Regression Model Accuracy: ", accuracy_score(y_test, y_pred_logreg))

Logistic Regression Model Accuracy:  0.7239


In [27]:
print("\n Classification Report: ")
print(classification_report(y_test, y_pred_logreg))


 Classification Report: 
              precision    recall  f1-score   support

           0       0.74      0.87      0.80      6366
           1       0.67      0.47      0.56      3634

    accuracy                           0.72     10000
   macro avg       0.71      0.67      0.68     10000
weighted avg       0.72      0.72      0.71     10000

